In [ ]:
import subprocess
import sys
import itertools
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()

sys.path.insert(0, str(PROJECT_ROOT))
from src.config import MODELS

def generate(name: str, regions: int, resolution: float, from_period: float | None = None, to_period: float | None = None, integers: bool = False, cap_fraction: float | None = None) -> int:
    args = [
        "esp", "generate",
        "--name", name,
        "--regions", str(regions),
        "--resolution", str(resolution),
    ]
    if from_period is not None:
        args += ["--from-period", str(from_period)]
    if to_period is not None:
        args += ["--to-period", str(to_period)]
    if integers:
        args += ["--integers"]
    if cap_fraction is not None:
        args += ["--cap-fraction", str(cap_fraction)]

    process = subprocess.Popen(
        args, cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    return process.returncode

In [ ]:
generateIntegers = False
CAP_FRACTION = None  # scale existing plant capacity; None → no reduction (GAMS default of 1.0)

periods = [
    (0, 7/365),  # single week
    (0, 1/12),   # one month
    (0.25, 0.5), # quarter
    (0.5, 1),    # half year
    (0, 1),      # full year
]

region_sizes = [3, 4, 5, 6, 7,        # small-scale
                9, 10, 11, 13, 15, 16, 17, 20,     # national / bidding-zone scale
                22, 23, 25, 27, 29, 30, 32, 34, 35]     # continental scale

# mip_test_regions = [5, 10, 25]

# region_sizes = mip_test_regions

resolutions = [1,    # hourly
               8,    # system default
               24, 168]
                # ,   # daily
            #    168]  # weekly

In [ ]:
failed = []
combinations = list(itertools.product(region_sizes, resolutions, periods))

for i, (regions, resolution, (from_p, to_p)) in enumerate(combinations, 1):
    suffix = "_mip_obj" if generateIntegers else "_obj"
    cf_tag = f"_cf{CAP_FRACTION}" if CAP_FRACTION is not None else ""
    name = f"r{regions}_res{resolution:.0f}_f{from_p:.4f}_t{to_p:.4f}{cf_tag}{suffix}"
    print(f"[{i}/{len(combinations)}] {name}", flush=True)
    if (MODELS / name / "original.mps").exists():
        print("  skipped (already exists)", flush=True)
        continue
    rc = generate(name, regions=regions, resolution=resolution, from_period=from_p, to_period=to_p,
                  integers=generateIntegers, cap_fraction=CAP_FRACTION)
    if rc != 0:
        failed.append(name)
        print(f"  FAILED: {name}")
    print(flush=True)

if failed:
    print(f"\n{len(failed)} model(s) failed: {failed}")
else:
    print(f"\nAll {len(combinations)} models generated successfully.")

[1/440] r3_res1_f0.0000_t0.0192_obj
